In [1]:
import pandas as pd
import re
import os
import matplotlib.pyplot as plt
import numpy as np

from IPython.display import display, HTML

In [2]:
execution_path = "../../../"

output_folder = (
    "SDDSmoteBoost_Unf_All_Patient_Direct_Base"  # TL + Base - Direct Base folder
)
output_folder = (
    "SDDSmoteBoost_Unf_All_Patient"  #  TL + Base - Pre-Saved Base Models folder
)


RESULT_DIR = f"{execution_path}/Output_Results/{output_folder}/"
print("RESULT_DIR", RESULT_DIR)

# Possible Values: "TL_1_Frozen", "TL_2_Frozen", "TL_3_Frozen", "TL_All_Frozen", "TL_All_Unfrozen"
Architecture_GROUP_List = [
    "TL_All_Unfrozen",
]

Model_Type_list = [
    "K-NN",
    "LogisticRegression",
    "NaiveBayes",
    "SVM",
    "DecisionTree",
]

patient_target_grp = [
    "SB-001",
    "SB-003",
    "SB-008",
    "SB-011",
    "SB-012",
    "SB-022",
    "SB-023",
    "SB-025",
    "SB-028",
    "SB-037",
    "SB-043",
    "SB-056",
    "SB-059",
    "SB-060",
    "SB-070",
    "SB-071",
    "SB-073",
    "SB-078",
    "SB-079",
    "SB-080",
    "SB-082",
    "SB-083",
    "SB-112",
]
print("patient_target_grp", patient_target_grp)

data_processing_columns = [
    "Patient_ID",
    "Pop_Epoch",
    "Pop_Fold",
    "Ind_Epoch",
    "Ind_Fold",
    "Model_Type",
]


# WHEN FALSE, TARGET DATA IS NOT REMOVED FROM ALL PATIENTS DATA
# False for Generalized model, True for Patient-wise Pop Models
bool_remove_target = False

# Number of Folds for running Pop Model
# Possible values : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
num_pop_folds = 5

# Number of Folds for running Ind Model
# Possible values : [3, 4, 5]
num_ind_folds = 3

POPULATION_EPOCH_SIZE_LIST = [100, 300, 500, 700, 1000, 2000, 3000]

PATIENT_EPOCH_SIZE_LIST = [100, 300, 500, 700, 1000, 2000, 3000]

RESULT_DIR ../../..//Output_Results/SDDSmoteBoost_Unf_All_Patient/
patient_target_grp ['SB-001', 'SB-003', 'SB-008', 'SB-011', 'SB-012', 'SB-022', 'SB-023', 'SB-025', 'SB-028', 'SB-037', 'SB-043', 'SB-056', 'SB-059', 'SB-060', 'SB-070', 'SB-071', 'SB-073', 'SB-078', 'SB-079', 'SB-080', 'SB-082', 'SB-083', 'SB-112']


In [3]:
def segregate_identifier(
    index,
    column_return_type,
    POPULATION_EPOCH_SIZE_LIST,
    PATIENT_EPOCH_SIZE_LIST,
    target_grp,
    num_pop_folds,
    num_ind_folds,
    Model_Type_list,
):
    index = str(index).upper()
    if column_return_type == "Pop_Epoch":
        for i in POPULATION_EPOCH_SIZE_LIST:
            if str(i) + "_POP" in index:
                return i
    elif column_return_type == "Ind_Epoch":
        for j in PATIENT_EPOCH_SIZE_LIST:
            if str(j) + "_IND" in index:
                return j
    elif column_return_type == "Patient_ID":
        for i in target_grp:
            if str(i) in index:
                return i
    elif column_return_type == "Pop_Fold":
        for i in [*range(1, num_pop_folds + 1)]:
            if "POP_FOLD_" + str(i) in index:
                return i
    elif column_return_type == "Ind_Fold":
        for i in [*range(1, num_ind_folds + 1)]:
            if "IND_FOLD_" + str(i) in index:
                return i
    elif column_return_type == "Model_Type":
        for i in Model_Type_list:
            if i.upper() in index:
                return i.upper()


def load_csv_to_df(
    eval_data_type,
    RESULT_DIR,
    Architecture_GROUP,
    bool_remove_target,
    data_processing_columns,
    POPULATION_EPOCH_SIZE_LIST,
    PATIENT_EPOCH_SIZE_LIST,
    target_grp,
    num_pop_folds,
    num_ind_folds,
    Model_Type_list,
):
    if bool_remove_target == True:
        suffix = "_evaluation_metrics_TargetRemoved.csv"
    else:
        suffix = "_evaluation_metrics.csv"

    if eval_data_type == "individual":
        eval_data_path = "/diagrams_n_individual_models/Ind_"
    elif eval_data_type == "population":
        eval_data_path = "/diagrams_n_population_models/Pop_"
    elif eval_data_type == "TL-Base":
        eval_data_path = "/diagrams_n_TL-Base_models/TL_Base_"

    file_path = (
        str(RESULT_DIR)
        + "/"
        + Architecture_GROUP
        + eval_data_path
        + Architecture_GROUP
        + suffix
    )

    if os.path.isfile(file_path):
        df = pd.read_csv(file_path, index_col=0)
        for i in data_processing_columns:
            df[i] = df.index.map(
                lambda x: segregate_identifier(
                    x,
                    i,
                    POPULATION_EPOCH_SIZE_LIST,
                    PATIENT_EPOCH_SIZE_LIST,
                    target_grp,
                    num_pop_folds,
                    num_ind_folds,
                    Model_Type_list,
                )
            )
        return df


bool_remove_target_list = [False]  # [True, False]
eval_data_type_list = ["individual", "population","TL-Base"]


All_eval_dict = {}

for i in eval_data_type_list:
    for j in bool_remove_target_list:
        for Architecture_GROUP in Architecture_GROUP_List:
            All_eval_dict[Architecture_GROUP + "_" + i + "_" + str(j)] = load_csv_to_df(
                i,
                RESULT_DIR,
                Architecture_GROUP,
                j,
                data_processing_columns,
                POPULATION_EPOCH_SIZE_LIST,
                PATIENT_EPOCH_SIZE_LIST,
                patient_target_grp,
                num_pop_folds,
                num_ind_folds,
                Model_Type_list,
            )

# del bool_remove_target_list
# del eval_data_type_list

In [4]:
display(All_eval_dict.keys())

pop_df = All_eval_dict["TL_All_Unfrozen_population_False"].copy(deep=True)
ind_df = All_eval_dict["TL_All_Unfrozen_individual_False"].copy(deep=True)
tl_base_df = All_eval_dict["TL_All_Unfrozen_TL-Base_False"].copy(deep=True)

cols_to_drop = ["Weighted Accuracy", "Specificity", "Precision_class0", "Precision_class1", "Precision_avg", "F1_class0", "F1_class1", "F1_avg", "auc_roc_score", "False_Discovery_Rate", "False_Negative_Rate", "False_Omission_Rate", "False_Positive_Rate", "Jaccard"]

# Drop the columns in place
pop_df.drop(columns=cols_to_drop, inplace=True)
ind_df.drop(columns=cols_to_drop, inplace=True)
tl_base_df.drop(columns=cols_to_drop, inplace=True)

# tl_base_df = tl_base_df[tl_base_df['Model_Type'] != 'DECISIONTREE']


dict_keys(['TL_All_Unfrozen_individual_False', 'TL_All_Unfrozen_population_False', 'TL_All_Unfrozen_TL-Base_False'])

In [5]:
# Creating a pivot table to aggregate 'Sensitivity/Recall' values for each 'Pop_Epoch'
pop_pivot_df = pop_df.pivot_table(index='Pop_Epoch', values='Sensitivity/Recall', aggfunc='mean')
pop_pivot_df.index = pop_df['Pop_Epoch'].unique()
pop_pivot_df.columns = ['Sensitivity/Recall']

display(pop_pivot_df)

pop_pivot_df.to_csv("pop_model_aggregated.csv")

pop_pivot_df_index_list = ["{} ({:.6f})".format(index, value) for index, value in zip(pop_pivot_df.index, pop_pivot_df['Sensitivity/Recall'])]

print(pop_pivot_df_index_list)

,Sensitivity/Recall
100,0.328661
300,0.239139
500,0.259347
700,0.243107
1000,0.252250
2000,0.248814
3000,0.240230


['100 (0.328661)', '300 (0.239139)', '500 (0.259347)', '700 (0.243107)', '1000 (0.252250)', '2000 (0.248814)', '3000 (0.240230)']


In [6]:
# Replace values in the 'Model_Type' column with "NN"
ind_df['Model_Type'] = ind_df['Model_Type'].replace(to_replace=ind_df['Model_Type'].unique(), value="NN")

display(ind_df)

,Sensitivity/Recall,Patient_ID,Pop_Epoch,Pop_Fold,Ind_Epoch,Ind_Fold,Model_Type
source-_size__5182_TL_All_Unfrozen_100_POP_FOLD_1_SB-001-100_IND_FOLD_1,0.250000,SB-001,100,1,100,1,NN
source-_size__5182_TL_All_Unfrozen_100_POP_FOLD_1_SB-001-100_IND_FOLD_2,0.545455,SB-001,100,1,100,2,NN
source-_size__5182_TL_All_Unfrozen_100_POP_FOLD_1_SB-001-100_IND_FOLD_3,0.357143,SB-001,100,1,100,3,NN
source-_size__5182_TL_All_Unfrozen_100_POP_FOLD_1_SB-001-300_IND_FOLD_1,0.333333,SB-001,100,1,300,1,NN
source-_size__5182_TL_All_Unfrozen_100_POP_FOLD_1_SB-001-300_IND_FOLD_2,0.181818,SB-001,100,1,300,2,NN
...,...,...,...,...,...,...,...
source-_size__5182_TL_All_Unfrozen_3000_POP_FOLD_5_SB-112-2000_IND_FOLD_2,0.222222,SB-112,3000,5,2000,2,NN
source-_size__5182_TL_All_Unfrozen_3000_POP_FOLD_5_SB-112-2000_IND_FOLD_3,0.200000,SB-112,3000,5,2000,3,NN
source-_size__5182_TL_All_Unfrozen_3000_POP_FOLD_5_SB-112-3000_IND_FOLD_1,0.166667,SB-112,3000,5,3000,1,NN
source-_size__5182_TL_All_Unfrozen_3000_POP_FOLD_5_SB-112-3000_IND_FOLD_2,0.222222,SB-112,3000,5,3000,2,NN


In [7]:
# Creating a pivot table to aggregate 'Sensitivity/Recall' values for each 'Pop_Epoch'
epoch_map = dict(zip(POPULATION_EPOCH_SIZE_LIST, pop_pivot_df_index_list))
ind_df['Pop_Epoch_NN'] = ind_df['Pop_Epoch'].replace(epoch_map)

# Creating a pivot table to aggregate 'Sensitivity/Recall' values for each 'Pop_Epoch' and 'Ind_Epoch' combination
ind_pivot_table = ind_df.pivot_table(index=['Pop_Epoch_NN', 'Model_Type'], columns='Ind_Epoch', values='Sensitivity/Recall', aggfunc='mean')

# ind_pivot_table.index = pivot_table_index_list
ind_pivot_table.columns = ind_df['Ind_Epoch'].astype(str).unique()

display(ind_pivot_table)

ind_pivot_table.to_csv("ind_model_aggregated.csv")

,,100,300,500,700,1000,2000,3000
Pop_Epoch_NN,Model_Type,,,,,,,
100 (0.328661),NN,0.462310,0.364774,0.354129,0.359251,0.349733,0.366560,0.375658
1000 (0.252250),NN,0.474844,0.409116,0.384025,0.376992,0.360307,0.383928,0.386684
2000 (0.248814),NN,0.467422,0.405868,0.393720,0.385614,0.378309,0.375568,0.382189
300 (0.239139),NN,0.476678,0.377163,0.361346,0.348788,0.348579,0.369372,0.371839
3000 (0.240230),NN,0.465554,0.396293,0.390549,0.380667,0.381278,0.375816,0.382072
500 (0.259347),NN,0.470782,0.395682,0.366325,0.346221,0.359303,0.381121,0.369325
700 (0.243107),NN,0.459122,0.389472,0.386151,0.370868,0.380666,0.382968,0.392040


In [8]:
epoch_map = dict(zip(POPULATION_EPOCH_SIZE_LIST, pop_pivot_df_index_list))
tl_base_df['Pop_Epoch_NN'] = tl_base_df['Pop_Epoch'].replace(epoch_map)

tl_base_pivot_df = tl_base_df.pivot_table(index=['Pop_Epoch_NN', 'Model_Type'], columns='Ind_Epoch', values='Sensitivity/Recall', aggfunc='mean')

tl_base_pivot_df.columns = tl_base_df['Ind_Epoch'].astype(str).unique()

# display(tl_base_pivot_df)

concat_df = pd.concat([tl_base_pivot_df,ind_pivot_table], axis=0)
concat_df = concat_df.sort_index(level=[0, 1])
# concat_df = concat_df.reindex(sorted(concat_df.index), axis=1)

display(concat_df)


concat_df.to_csv("tl_base_model_aggregated.csv")

100       300       500       700  \
Pop_Epoch_NN    Model_Type                                                   
100 (0.328661)  K-NN                0.690934  0.779562  0.795473  0.801457   
                LOGISTICREGRESSION  0.674779  0.748908  0.757786  0.760332   
                NAIVEBAYES          0.656217  0.758099  0.770896  0.780291   
                NN                  0.462310  0.364774  0.354129  0.359251   
                SVM                 0.650965  0.734216  0.754097  0.756407   
1000 (0.252250) K-NN                0.683195  0.768155  0.786658  0.793896   
                LOGISTICREGRESSION  0.669826  0.731113  0.747854  0.746053   
                NAIVEBAYES          0.629589  0.722390  0.741656  0.750714   
                NN                  0.474844  0.409116  0.384025  0.376992   
                SVM                 0.647381  0.717701  0.732264  0.736039   
2000 (0.248814) K-NN                0.656702  0.736510  0.762844  0.770492   
                LOGISTICREGRESSION  0.668171  0.719368  0.727167  0.733662   
                NAIVEBAYES          0.597628  0.699134  0.718020  0.725318   
                NN                  0.467422  0.405868  0.393720  0.385614   
                SVM                 0.602239  0.691091  0.700705  0.708663   
300 (0.239139)  K-NN                0.677536  0.775959  0.793836  0.797308   
                LOGISTICREGRESSION  0.669106  0.737925  0.750405  0.752905   
                NAIVEBAYES          0.637899  0.737524  0.762998  0.767476   
                NN                  0.476678  0.377163  0.361346  0.348788   
                SVM                 0.647515  0.719403  0.739259  0.748517   
3000 (0.240230) K-NN                0.661441  0.737075  0.766867  0.773455   
                LOGISTICREGRESSION  0.662846  0.716317  0.730936  0.734216   
                NAIVEBAYES          0.605778  0.696445  0.719585  0.733101   
                NN                  0.465554  0.396293  0.390549  0.380667   
                SVM                 0.609544  0.666202  0.699635  0.701435   
500 (0.259347)  K-NN                0.694669  0.771684  0.787862  0.793839   
                LOGISTICREGRESSION  0.669928  0.729060  0.741790  0.746172   
                NAIVEBAYES          0.632607  0.733892  0.748363  0.754603   
                NN                  0.470782  0.395682  0.366325  0.346221   
                SVM                 0.626411  0.717926  0.731081  0.732581   
700 (0.243107)  K-NN                0.692805  0.772181  0.792617  0.793998   
                LOGISTICREGRESSION  0.679022  0.734104  0.747248  0.746976   
                NAIVEBAYES          0.645181  0.731902  0.753723  0.759930   
                NN                  0.459122  0.389472  0.386151  0.370868   
                SVM                 0.637081  0.721889  0.733789  0.736236   

                                        1000      2000      3000  
Pop_Epoch_NN    Model_Type                                        
100 (0.328661)  K-NN                0.804226  0.812730  0.813073  
                LOGISTICREGRESSION  0.766381  0.773178  0.771895  
                NAIVEBAYES          0.779708  0.790210  0.791804  
                NN                  0.349733  0.366560  0.375658  
                SVM                 0.761398  0.766757  0.771157  
1000 (0.252250) K-NN                0.789731  0.804642  0.802088  
                LOGISTICREGRESSION  0.748058  0.749742  0.747424  
                NAIVEBAYES          0.750036  0.765074  0.761947  
                NN                  0.360307  0.383928  0.386684  
                SVM                 0.737751  0.743900  0.735638  
2000 (0.248814) K-NN                0.778965  0.785675  0.791126  
                LOGISTICREGRESSION  0.741014  0.738598  0.742301  
                NAIVEBAYES          0.736417  0.743288  0.740159  
                NN                  0.378309  0.375568  0.382189  
                SVM                 0.716889  0.722977  0.715109  
300 (0.239139) 